In [1]:
# This notebook aim to be a step by step illustration of Abench application.

import os, warnings
import numpy as np
import time

import numpy as np
import pandas as pd
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0,'../../uqmodels/abench')
import abench 
sys.path.insert(0,'../../uqmodels/')
import uqmodels
sys.path.insert(0,'src/')


import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
tf.config.set_logical_device_configuration(gpus[0],[tf.config.LogicalDeviceConfiguration(memory_limit=4096)])

2026-02-03 14:08:03.860882: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770127683.879954    1751 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770127683.885451    1751 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-03 14:08:03.904421: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd


def make_base_df(n: int = 500, seed: int = 0) -> pd.DataFrame:
    """Base dataframe with stable schema, types, missingness and distributions."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame(
        {
            "a": rng.normal(loc=0.0, scale=1.0, size=n).astype(float),
            "b": rng.integers(low=0, high=100, size=n).astype(int),
            "c": rng.choice(["x", "y", "z"], size=n, p=[0.6, 0.3, 0.1]).astype(object),
        }
    )


def generate_4_csvs(
    out_dir: Path,
    *,
    n_clean: int = 8,
    n_rows: int = 500,
    seed: int = 123,
) -> pd.DataFrame:
    """
    Create a small synthetic dataset folder containing:
      - `n_clean` healthy CSV files (same schema, types, missingness, distributions)
      - 4 "faulty" CSV files, each violating a different criterion:
          1) columns: missing column 'c'
          2) dtypes: column 'b' stored as string/object
          3) missingness: column 'a' has ~50% NaNs
          4) distribution: column 'a' shifted by +6

    Parameters
    ----------
    out_dir:
        Output directory where CSVs will be written.
    n_clean:
        Number of healthy CSV files to generate.
    n_rows:
        Number of rows per CSV.
    seed:
        Random seed controlling the clean files generation.

    Returns
    -------
    index_df:
        DataFrame with columns: id, path
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)

    paths = []
    ids = []

    # --- clean files --------------------------------------------------------
    for i in range(n_clean):
        df = make_base_df(n=n_rows, seed=int(rng.integers(0, 10_000_000)))
        p = out_dir / f"clean_{i:03d}.csv"
        df.to_csv(p, index=False)
        ids.append(f"clean_{i:03d}")
        paths.append(str(p))

    # --- faulty 1: missing column 'c' --------------------------------------
    df_cols = make_base_df(n=n_rows, seed=int(rng.integers(0, 10_000_000))).drop(columns=["c"])
    p1 = out_dir / "err_columns_missing_c.csv"
    df_cols.to_csv(p1, index=False)
    ids.append("err_columns")
    paths.append(str(p1))

    # --- faulty 2: dtype mismatch (b as string) ----------------------------
    df_dtypes = make_base_df(n=n_rows, seed=int(rng.integers(0, 10_000_000)))
    df_dtypes["b"] = df_dtypes["b"].astype(str)
    p2 = out_dir / "err_dtypes_b_as_string.csv"
    df_dtypes.to_csv(p2, index=False)
    ids.append("err_dtypes")
    paths.append(str(p2))

    # --- faulty 3: missingness shift (50% NaNs in 'a') ---------------------
    df_nan = make_base_df(n=n_rows, seed=int(rng.integers(0, 10_000_000)))
    mask = rng.random(len(df_nan)) < 0.5
    df_nan.loc[mask, "a"] = np.nan
    p3 = out_dir / "err_missingness_a_nan50.csv"
    df_nan.to_csv(p3, index=False)
    ids.append("err_missingness")
    paths.append(str(p3))

    # --- faulty 4: distribution shift (a shifted by +6) --------------------
    df_dist = make_base_df(n=n_rows, seed=int(rng.integers(0, 10_000_000)))
    df_dist["a"] = df_dist["a"] + 6.0
    p4 = out_dir / "err_distribution_a_shifted.csv"
    df_dist.to_csv(p4, index=False)
    ids.append("err_distribution")
    paths.append(str(p4))

    return pd.DataFrame({"id": ids, "path": paths})
    
if True :
    # Generate Test data
    from pathlib import Path
    out_dir = Path("demo")
    index_df = generate_4_csvs(out_dir, n_clean=10, n_rows=500, seed=0)

In [7]:
from abench.store.data_integrity import AuditContext,build_default_auditor

from abench.store.data_management import explore_csv_hierarchy
index_df = explore_csv_hierarchy('demo',)

ctx = AuditContext(
    n_bins=10,
    alpha=0.01,
    max_js_div=0.05,              # make distribution test more sensitive for the demo
    required_presence_ratio=0.75,  # "required" if present in >= 3/4 files
    allow_extra_columns=True,
    read_csv_kwargs={})

auditor = build_default_auditor(ctx)
report = auditor.run(index_df, path_col="path", key_col="id")

print("=== SUMMARY ===")
print(report["summary"])

print("\n=== PER FILE STATUS ===")
for file_id, entry in report["files"].items():
    g = entry.get("diagnostic", {}).get("global_status")
    crit_status = {k: v.get("status") for k, v in entry.get("diagnostic", {}).items() if isinstance(v, dict) and "status" in v}
    path = report["files"][file_id]['path']
    print(f"- {path} global={g}  criteria={crit_status}")

# Optional: show details for each failing file
print("\n=== FAIL DETAILS (compact) ===")
for file_id, entry in report["files"].items():
    if entry.get("diagnostic", {}).get("global_status") != "KO":
        continue
    print(f"\n[{file_id}] path={entry['path']}")
    for crit_name in ["columns", "dtypes", "missingness", "distribution"]:
        diag = entry["diagnostic"].get(crit_name, {})
        if diag.get("status") == "KO":
            print(f"  - {crit_name}: KO")
            # print only a small subset
            if crit_name == "columns":
                print("    ", diag.get("details"))
            else:
                # show first 5 KO columns (if any)
                per_col = diag.get("per_column", {})
                ko_cols = [c for c, s in per_col.items() if s == "KO"]
                print(f"    KO columns (first 5): {ko_cols[:5]}")

=== SUMMARY ===
{'n_ok': 11, 'n_ko': 3}

=== PER FILE STATUS ===
- demo/clean_000.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution': 'OK'}
- demo/clean_001.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution': 'OK'}
- demo/clean_002.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution': 'OK'}
- demo/clean_003.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution': 'OK'}
- demo/clean_004.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution': 'OK'}
- demo/clean_005.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution': 'OK'}
- demo/clean_006.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution': 'OK'}
- demo/clean_007.csv global=OK  criteria={'columns': 'OK', 'dtypes': 'OK', 'missingness': 'OK', 'distribution